In [1]:
from sedona.spark import SedonaContext
import os
from time import time
import pyspark.sql.functions as f

In [2]:
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

In [5]:
def create_session(index_type: str, number_partitions: int):
    config_params =  {
        "spark.driver.memory": "12G",
        "spark.executor.memory": "16G",
        "sedona.sql.additional.functions": "org.sedona.sql.ST_AreaAdditional",
        "sedona.join.gridtype": index_type,
        "sedona.join.numpartition": number_partitions
    }

    config = SedonaContext.builder()
    
    for key, value in config_params.items():
        config = config.config(key, value)
    
    sedona = SedonaContext.create(config.getOrCreate())
    sedona.sparkContext.setLogLevel("ERROR")

    return sedona

In [3]:
def do_spatial_join(sedona, index_type: str, number_partitions: int, repeats: int):
    buildings = sedona.\
        read.\
        format("geoparquet").\
        load(f"s3a://{bucket_name}/source_data/optimizations/buildings.geoparquet")
    
    places = sedona.\
        read.\
        format("geoparquet").\
        load(f"s3a://{bucket_name}/source_data/optimizations/places.geoparquet")

    sum_time = 0
    for _ in range(repeats):
        start = time()

        buildings.alias("b").\
            join(places.alias("p"), f.expr("ST_DWithin(b.geometry, p.geometry, 0.001)")).\
            count()

        sum_time += time() - start
        
    partition_df = sedona.read.option("delimiter", ";").format("csv").\
        load(os.path.join(path, f"{index_type}_{number_partitions}")).\
        selectExpr("_c0 AS id", "ST_GeomFromText(_c1) AS geometry")

    buildings_stats = buildings.alias("p1").join(partition_df.alias("p2"), f.expr("ST_Intersects(p1.geometry, p2.geometry)")).\
        groupBy("p2.id").\
        count()

    stddev_buildings = buildings_stats.\
        selectExpr("stddev(count)").\
        collect()[0][0]

    places_stats = places.alias("p1").join(partition_df.alias("p2"), f.expr("ST_Intersects(p1.geometry, p2.geometry)")).\
        groupBy("p2.id").\
        count()

    stddev_places = places_stats.\
        selectExpr("stddev(count)").\
        collect()[0][0]

    diff_stdev = buildings_stats.alias("b").join(
        places_stats.alias("p"), ["id"]
    ).selectExpr("(b.count/p.count)*100 AS diff").\
        selectExpr("stddev(diff)").\
        collect()[0][0]

    return {
        "stddev_buildings": stddev_buildings,   
        "stddev_places": stddev_places,
        "diff_stdev": diff_stdev,
        "avg_time": sum_time/repeats
    }
    

In [ ]:
partition_nums = [10, 20, 40, 100, 200]
index_types = ["kdbtree", "quadtree"]

all_times = {
    "kdbtree": [],
    "quadtree": [],
}
for p in partition_nums:
    for i in index_types:
        print(f"p {p} i: {i}")
        sedona = create_session(i, p)
        result = do_spatial_join(sedona, i, p, 3)

        all_times[i].append(result)
        
        sedona.stop()

In [3]:
config = SedonaContext.builder().\
    config("spark.executor.memory", "12G").\
    config("spark.driver.memory", "16G").\
    config("sedona.join.autoBroadcastJoinThreshold", "-1").\
    config("spark.hadoop.fs.s3a.multipart.size", "24M").\
    config("spark.hadoop.fs.s3a.threads.max", "64").\
    config("spark.hadoop.fs.s3a.connection.maximum", "1000")

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/24 22:21:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/24 22:21:35 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/24 22:21:35 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/24 22:21:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/24 22:21:35 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/08/24 22:21:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/24 22:21:35 WARN SimpleFunctionRegistry: The function st_envelop

In [11]:
result_data = []
partition_nums = [10, 20, 40, 100, 200]
for key, value in all_times.items():
    index = 0
    for el in value:
        result_data.append(
            {
                "partitioning": key,
                "num_partitions": partition_nums[index],
                **el
                
            }
        )
        index+=1

In [12]:
metrics = sedona.createDataFrame(result_data).\
    selectExpr(
        "partitioning", 
        "num_partitions",
        "ROUND(avg_time, 2) AS avg_time",
        "ROUND(diff_stdev, 2) AS diff_stdev",
        "ROUND(stddev_buildings, 2) AS stddev_buildings",
        "ROUND(stddev_places, 2) AS stddev_places",
    )

In [13]:
metrics.createOrReplaceTempView("metrics")

In [14]:
sedona.sql(
"""
WITH pivot AS (
    SELECT 
        * 
    FROM metrics
    PIVOT (
        MAP(
            'avg_time', FIRST(avg_time),
            'diff_stdev', FIRST(diff_stdev),
            'stddev_buildings', FIRST(stddev_buildings),
            'stddev_places', FIRST(stddev_places)
        ) AS elements
        FOR partitioning IN ('kdbtree' AS num_kdbtree, 'quadtree' AS num_quadtree)
    )
), exploded AS(
SELECT 
    num_partitions,
    explode(num_kdbtree),
    num_quadtree
FROM pivot
) 
SELECT 
    num_partitions,
    key AS measure,
    value AS kdbtree_value,
    num_quadtree[key] AS quadtree_value
FROM exploded
"""
).show()

+--------------+----------------+-------------+--------------+
|num_partitions|         measure|kdbtree_value|quadtree_value|
+--------------+----------------+-------------+--------------+
|            10|        avg_time|          5.0|          NULL|
|            10|      diff_stdev|          3.0|          NULL|
|            10|stddev_buildings|          1.0|          NULL|
|            10|   stddev_places|          2.0|          NULL|
+--------------+----------------+-------------+--------------+

